### Analysing stage 4 CKD characteristics ###

We have two tables:
  * **ckd_enrollment**: Information about members
  * **ckd_claims**: Their claims (these include the combination of inpatient, outpatient and Rx claims)

The data spans the years 2017, 2018 and 2019.

In [1]:
# -----------------------------------------------------------------------------
# INITIALIZATION
# -----------------------------------------------------------------------------
import sys
print(f"Python version: {sys.version}")

import logging
import csv
import gzip
import re
import pandas as pd
import numpy as np
from functools import reduce

import pyspark
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql import SparkSession
from pyspark import SparkConf
import plotly.express as px
import plotly.graph_objects as go
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

# -----------------------------------------------------------------------------
# INITIALIZE LOGGING
# -----------------------------------------------------------------------------
f = '%(asctime)-15s %(levelname)-8s %(message)s'
logger = logging.getLogger(__name__)
logger.setLevel("DEBUG")
logging.basicConfig(format=f)


from IPython.core.magic import register_cell_magic

# -----------------------------------------------------------------------------
# start_spark
# -----------------------------------------------------------------------------
def start_spark(
    driver_memory="100g",
    storage_fraction=0.5,
    num_nodes=10,
):
    """Initialize spark

    Arguments:
        driver_memory: Maximum heap size for the Spark driver Java
            virtual machine.
        storage_fraction: Controls what portion of Spark's unified
            memory is reserved for storage (i.e., caching/persisting data
            and broadcast variables), as a fraction of the total
            execution + storage memory pool.
            If you cache/persist a lot of data, and you're evicting
            data too early, you might increase this value (e.g. 0.6 or 0.7).
            Conversely, if your job is shuffle-heavy and fails due to
            memory pressure, you might decrease it (e.g. 0.3).
        num_nodes: How many concurrent threads to use while running
            in "local mode" (i.e. in a single machine instead of a cluster).
            Use '*' to use all cores, or an integer > 0 for a specific
            number of threads.
    """

    conf = SparkConf().setAppName("My_Application")
    conf.set("spark.driver.memory", driver_memory)
    conf.set("spark.memory.storageFraction", str(storage_fraction))
    conf.setMaster(f"local[{num_nodes}]")

    spark = SparkSession.builder.config(conf=conf).getOrCreate()
    spark.sparkContext.setLogLevel('WARN')

    return spark


Python version: 3.11.0 (main, Jun 13 2025, 14:48:45) [Clang 16.0.0 (clang-1600.0.26.6)]


In [2]:

spark = start_spark(num_nodes=10)
#spark.stop()

  
@register_cell_magic
def spark_sql(line, cell):
    result = spark.sql(cell)
    result.show(n=1000)
  

# -- READ ENROLLMENT AND DATA TABLES
enrollment_file = f"/Users/Charles/DATA/ckd/ckd_enrollment"
logger.info(f">>> Reading enrollment file: {enrollment_file}")
df_enrollment = spark.read.format("parquet").load(enrollment_file)
df_enrollment.createOrReplaceTempView('enrollment')
logger.info(f">>> ENROLLMENT has {df_enrollment.count():,} rows")
logger.info(f">>> ENROLLMENT has {df_enrollment.select('ENROLID').distinct().count():,} unique enrollees")

claims_file = f"/Users/Charles/DATA/ckd/ckd_claims"
logger.info(f">>> Reading claims file: {claims_file}")
df_claims = spark.read.format("parquet").load(claims_file)
df_claims.createOrReplaceTempView('claims')
logger.info(f">>> CLAIMS has {df_claims.count():,} rows")
logger.info(f">>> CLAIMS has {df_claims.select('ENROLID').distinct().count():,} unique enrollees")

# -- NOTE: with the "createOrReplaceTempView" we define a view of these
# -- tables, so we can use them in SQL queries.

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/11 10:26:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2025-07-11 10:26:09,562 INFO     >>> Reading enrollment file: /Users/Charles/DATA/ckd/ckd_enrollment
25/07/11 10:26:10 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
2025-07-11 10:26:11,157 INFO     >>> ENROLLMENT has 2,586,930 rows
2025-07-11 10:26:12,671 INFO     >>> ENROLLMENT has 862,310 unique enrollees    
2025-07-11 10:26:12,671 INFO     >>> Reading claims file: /Users/Charles/DATA/ckd/ckd_claims
2025-07-11 10:26:13,577 INFO     >>> CLAIMS has 253,070,875 rows                
2025-07-11 10:26:20,486 INFO     >

In [3]:
dx_cols = [col for col in df_claims.columns if col.__contains__("DX")]
# Step 2: Create CKD_STAGE using dynamic matching
def build_ckd_stage_case(dx_cols):
    stage_expr = None
    stage_map = {
        "N181": "CKD Stage 1",
        "N182": "CKD Stage 2",
        "N183": "CKD Stage 3",
        "N184": "CKD Stage 4",
        "N185": "CKD Stage 5",
        "N186": "ESRD",
        "N189": "Unspecified"
    }

    for code, label in stage_map.items():
        condition = reduce(lambda a, b: a | b, [F.col(c).startswith(code) for c in dx_cols])
        if stage_expr is None:
            stage_expr = F.when(condition, label)
        else:
            stage_expr = stage_expr.when(condition, label)
    if stage_expr is None:
        stage_expr = F.lit(None)
    return stage_expr

# Step 3: Filter for CKD codes and extract stage
ckd_filter = reduce(lambda a, b: a | b, [F.col(c).startswith("N18") for c in dx_cols])
ckd_stage_col = build_ckd_stage_case(dx_cols)
df_ckd_claims = (
    df_claims
    .filter(ckd_filter)
    .withColumn("CKD_STAGE", ckd_stage_col)
    .filter(F.col("CKD_STAGE").isNotNull())
    .filter(F.col("SVCDATE").isNotNull())
)


## STDPLAC by CKD stage
Next we analyze whether CKD Stage 4 patients are more likely to be seen in hospitals or specialty care settings, we’ll leverage the STDPLAC field in your claims data.

In [12]:
df_check = (
    df_ckd_claims
    .filter(F.col("CKD_STAGE") == "CKD Stage 1")
    .withColumn("Location", stdplac_map_expr.getItem(F.col("STDPLAC").cast("string")))
    .withColumn("Region_Label", region_map_expr.getItem(F.col("REGION").cast("string")))
    .filter((F.col("Location") == "Residential Facility") & (F.col("Region_Label") == "North Central"))
)
df_check.count()


25

In [15]:

df_check_pd = df_check.select("NETPAY").toPandas()
fig = px.histogram(
    df_check_pd,
    x="NETPAY",
    nbins=50,
    title="Distribution of NETPAY for CKD Stage 1 - Residential Facility (North Central)",
    labels={"NETPAY": "Net Payment ($)"}
)

fig.update_layout(bargap=0.1)
fig.show()

In [13]:
df_check.select("NETPAY").describe().show()


+-------+------------------+
|summary|            NETPAY|
+-------+------------------+
|  count|                25|
|   mean|1201.1876000000002|
| stddev|1278.0271210192163|
|    min|               0.0|
|    max|            3500.0|
+-------+------------------+



##### Load the json file and plot histogram

In [17]:
import plotly.express as px
import json
# Convert maps to PySpark expression
from pyspark.sql.functions import create_map, lit
from itertools import chain

# Load region labels and stdplac mappings
with open("geo_industry_map.json", "r") as f:
    mappings = json.load(f)
region_labels = mappings["region_labels"]

with open("stdplac_group_map.json", "r") as f:
    stdplac_group_map = json.load(f)



stdplac_map_expr = create_map([lit(k) for k in chain(*stdplac_group_map.items())])
region_map_expr = create_map([lit(k) for k in chain(*region_labels.items())])

# Main plotting function
def plot_ckd_stage_costs(df_ckd_claims, stages=["CKD Stage 1", "CKD Stage 2", "CKD Stage 3", "CKD Stage 4", "CKD Stage 5", "ESRD"]):
    plots = []

    for stage in stages:
        # Filter stage
        df_stage = df_ckd_claims.filter(F.col("CKD_STAGE") == stage)

        # Add mapped columns
        df_stage = (
            df_stage
            .withColumn("Location", stdplac_map_expr.getItem(F.col("STDPLAC").cast("string")))
            .withColumn("Region_Label", region_map_expr.getItem(F.col("REGION").cast("string")))
            .fillna({"Location": "Unmapped", "Region_Label": "Unknown"})
        )
        df_stage_pd = df_stage.toPandas()

        # Special handling for ESRD: remove the top outlier in Unknown region   
        if stage == "ESRD":
            # Remove top outlier row (max NETPAY) in Unknown region
            is_unknown = df_stage_pd["Region_Label"] == "Unknown"
            df_stage_pd = df_stage_pd[~((is_unknown))]
        
        # Group and aggregate
        df_summary = (
            df_stage.groupBy("Region_Label", "Location")
            .agg(
                F.sum("NETPAY").alias("Total_Cost"),
                F.count("*").alias("n_claims"),
                F.mean("NETPAY").alias("Avg_Cost_per_Claim")
            )
            .orderBy(F.desc("Avg_Cost_per_Claim"))
        )

        # Convert to pandas for plotting
        df_summary_pd = df_summary.toPandas()

        # Plot
        fig = px.bar(
            df_summary_pd,
            x="Location",
            y="Avg_Cost_per_Claim",
            color="Region_Label",
            barmode="group",
            title=f"Average {stage} Cost per Claim by Region and Visit Setting",
            labels={
                "Avg_Cost_per_Claim": "Average Cost per Claim ($)",
                "Location": "Visit Setting",
                "Region_Label": "Region"
            },
            height=600,
            width=1000
        )
        fig.update_layout(xaxis_tickangle=-45)
        fig.show()
        plots.append(fig)

    return plots

plots = plot_ckd_stage_costs(df_ckd_claims)


/Users/cat2510/.pyenv/versions/3.11.0/envs/analytics/lib/python3.11/site-packages/pyspark/sql/classic/column.py:359: FutureWarning:

A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.



/Users/cat2510/.pyenv/versions/3.11.0/envs/analytics/lib/python3.11/site-packages/pyspark/sql/classic/column.py:359: FutureWarning:

A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.



/Users/cat2510/.pyenv/versions/3.11.0/envs/analytics/lib/python3.11/site-packages/pyspark/sql/classic/column.py:359: FutureWarning:

A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.



/Users/cat2510/.pyenv/versions/3.11.0/envs/analytics/lib/python3.11/site-packages/pyspark/sql/classic/column.py:359: FutureWarning:

A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.



/Users/cat2510/.pyenv/versions/3.11.0/envs/analytics/lib/python3.11/site-packages/pyspark/sql/classic/column.py:359: FutureWarning:

A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.



/Users/cat2510/.pyenv/versions/3.11.0/envs/analytics/lib/python3.11/site-packages/pyspark/sql/classic/column.py:359: FutureWarning:

A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.



In [ ]:
import pandas as pd
from pyspark.sql import functions as F

def summarize_ckd_stage_costs(df_ckd_claims, stdplac_group_map, region_labels,
                               stages=["CKD Stage 1", "CKD Stage 2", "CKD Stage 3", "CKD Stage 4", "CKD Stage 5", "ESRD"]):
    all_stats = []

    for stage in stages:
        df_stage = df_ckd_claims.filter(F.col("CKD_STAGE") == stage)

        # Create Spark map expressions
        #stdplac_map_expr = F.create_map([x for kv in stdplac_group_map.items() for x in (F.lit(kv[0]), F.lit(kv[1]))])
        region_map_expr = F.create_map([x for kv in region_labels.items() for x in (F.lit(kv[0]), F.lit(kv[1]))])

        # Apply mappings
        df_stage = (
            df_stage
           # .withColumn("Location", stdplac_map_expr.getItem(F.col("STDPLAC").cast("string")))
            .withColumn("Region_Label", region_map_expr.getItem(F.col("REGION").cast("string")))
            .fillna({ "Region_Label": "Unknown"}) #"Location": "Unmapped",
        )

        # Convert to Pandas
        df_stage_pd = df_stage.select("NETPAY", "Region_Label").toPandas() #, "Location"

        # Remove top outlier if ESRD + Unknown
        if stage == "ESRD":
            is_unknown = df_stage_pd["Region_Label"] == "Unknown"
            max_netpay = df_stage_pd[is_unknown]["NETPAY"].max()
            df_stage_pd = df_stage_pd[~((is_unknown) & (df_stage_pd["NETPAY"] == max_netpay))]

        # Aggregate summary stats
        summary = (
            df_stage_pd
            .groupby(["Region_Label"]) # , "Location"
            .agg(
                total_cost=pd.NamedAgg(column="NETPAY", aggfunc="sum"),
                n_claims=pd.NamedAgg(column="NETPAY", aggfunc="count"),
                avg_cost=pd.NamedAgg(column="NETPAY", aggfunc="mean"),
                std_cost=pd.NamedAgg(column="NETPAY", aggfunc="std"),
                max_cost=pd.NamedAgg(column="NETPAY", aggfunc="max"),
                min_cost=pd.NamedAgg(column="NETPAY", aggfunc="min"),
            )
            .reset_index()
        )
        summary["CKD_Stage"] = stage
        all_stats.append(summary)

    # Concatenate all stages
    result_df = pd.concat(all_stats, ignore_index=True)
    return result_df

summary_df = summarize_ckd_stage_costs(df_ckd_claims, stdplac_group_map, region_labels)
summary_df.head()


In [25]:
summary_df.sort_values(by="avg_cost",ascending=False).head(15)

,Region_Label,total_cost,n_claims,avg_cost,std_cost,max_cost,min_cost,CKD_Stage
26,Northeast,3.846135e+08,1034534,371.774666,3300.374039,882441.40,-341107.23,ESRD
25,North Central,3.771460e+08,1544271,244.222681,1863.711558,222235.33,-380902.78,ESRD
29,West,2.475402e+08,1029221,240.512222,2548.327641,607116.46,-601864.22,ESRD
28,Unknown,9.704947e+06,44011,220.511855,791.549994,102978.93,-15142.05,ESRD
27,South,9.451195e+08,4517099,209.231519,1718.468436,1134930.70,-747211.48,ESRD
20,North Central,7.842650e+06,41473,189.102556,1814.449644,149176.51,-67418.93,CKD Stage 5
21,Northeast,7.180178e+06,39742,180.669761,1555.242390,99999.00,-78590.00,CKD Stage 5
22,South,1.650970e+07,107708,153.281998,2047.657807,175891.28,-148712.40,CKD Stage 5
16,Northeast,1.296924e+07,85278,152.081864,2687.573267,252422.56,-252422.51,CKD Stage 4
24,West,4.599730e+06,33652,136.685203,1267.118890,130943.50,-27924.84,CKD Stage 5


In [27]:
import plotly.express as px

fig = px.bar(
    summary_df,
    x="Region_Label",
    y="avg_cost",
    color="CKD_Stage",
    barmode="group",
    title="Average Cost per Claim by Region and CKD Stage",
    labels={
        "avg_cost": "Average Cost ($)",
        "Region_Label": "Region",
        "CKD_Stage": "CKD Stage"
    },
    height=600,
    width=1000
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()


## Check comorbidity profiles of CKD Stage 4 patients
 using diagnosis codes

In [26]:
from pyspark.sql.functions import col, explode, array
df_check = (
    df_ckd_claims
    .filter(F.col("CKD_STAGE") == "CKD Stage 4")#.filter(F.col("YEAR") == "2017")
)
# Combine DX1-DX4 into one column for diagnosis frequency
df_dxs = df_check.select(
    explode(array("DX1", "DX2", "DX3", "DX4", "DXVER", "PDX")).alias("DX")
).filter(col("DX").isNotNull()& (col("DX") != "") & (col("DX") != "0") & ~(col("DX").startswith("N18")))

In [27]:
from pyspark.sql.functions import col



# Step 2: Count and convert to pandas
dx_counts_pd = (
    df_dxs.groupBy("DX")
    .count()
    .orderBy("count", ascending=False)
    .limit(20)
    .toPandas()
)


In [28]:
import json 
with open("icd_codes.json", "r") as f:
    data = json.load(f)

icd10_label_map = data["icd10_label_map"]
dx_counts_pd["DX_Label"] = dx_counts_pd["DX"].map(icd10_label_map).fillna(dx_counts_pd["DX"])
fig = px.bar(
    dx_counts_pd,
    x="count",
    y="DX_Label",
    orientation="h",
    title="Top Diagnosis Codes Among CKD Stage 4 Patients",
    labels={"count": "Number of Occurrences", "DX_Label": "Diagnosis"}
)
fig.update_layout(yaxis=dict(categoryorder="total ascending"))
fig.update_layout(
    yaxis=dict(
        tickmode='linear',
        dtick=1
    )
)
fig.show()

In [29]:
dx_counts_pd["DX"].tolist()

['D631',
 'I129',
 'I10',
 'E1122',
 'N179',
 'N2581',
 'E559',
 'D649',
 'R809',
 'E785',
 'E119',
 'I130',
 'E875',
 'E872',
 'Z01818',
 'D509',
 'E1121',
 'Z940',
 'E782',
 'Z79899']

##### create the json files containing the mapping from STDPLAC

In [ ]:
import json

# Updated mapping based on provided STDPLAC labels
updated_stdplac_map = {
    1: "Pharmacy",
    2: "Telehealth",
    3: "School",
    4: "Homeless Shelter",
    5: "Indian Hlth Svc Free-stand Fac",
    6: "Indian Hlth Svc Prov-based Fac",
    7: "Tribal 638 Free-standing Fac",
    8: "Tribal 638 Provider-based Fac",
    9: "Prison-Correctional Facility",
    10: "Telehealth Provided in Pat Hm",
    11: "Office",
    12: "Patient Home",
    13: "Assisted Living Facility",
    14: "Group Home",
    15: "Mobile Unit",
    16: "Temporary Lodging",
    17: "Walk-in Retail Health Clinic",
    18: "Place of Employment-Worksite",
    19: "Outpatient Hospital-Off Campus",
    20: "Urgent Care Facility",
    21: "Inpatient Hospital",
    22: "Outpatient Hospital-On Campus",
    23: "Emergency Room - Hospital",
    24: "Ambulatory Surgical Center",
    25: "Birthing Center",
    26: "Military Treatment Facility",
    27: "Outreach Site/Street",
    28: "Other Inpatient Care (NEC)",
    31: "Skilled Nursing Facility",
    32: "Nursing Facility",
    33: "Custodial Care Facility",
    34: "Hospice",
    35: "Adult Living Care Facility",
    41: "Ambulance (land)",
    42: "Ambulance (air or water)",
    49: "Independent Clinic",
    50: "Federally Qualified Health Ctr",
    51: "Inpatient Psychiatric Facility",
    52: "Psych Facility Partial Hosp",
    53: "Community Mental Health Center",
    54: "Intermed Care/Intellect Disab",
    55: "Residential Subst Abuse Facil",
    56: "Psych Residential Treatmnt Ctr",
    57: "Non-resident Subst Abuse Facil",
    60: "Mass Immunization Center",
    61: "Comprehensive Inpt Rehab Fac",
    62: "Comprehensive Outpt Rehab Fac",
    65: "End-Stage Renal Disease Facil",
    71: "State/Local Public Health Clin",
    72: "Rural Health Clinic",
    81: "Independent Laboratory",
    95: "Outpatient (NEC)",
    98: "Pharmacy",
    99: "Other/Unknown"
}

# Save to JSON
output_path = "stdplac_map.json"
with open(output_path, "w") as f:
    json.dump(updated_stdplac_map, f, indent=4)


# Load the corrected stdplac_map
with open("stdplac_map.json", "r") as f:
    stdplac_map = json.load(f)

# Create the grouping
stdplac_group_map = {}

# Mapping logic based on keywords
hospital_terms = ['Inpatient Hospital', 'Outpatient Hospital-Off Campus', 'Outpatient Hospital-On Campus', 'Emergency Room - Hospital']
clinic_terms = ['Office', 'Independent Clinic', 'Federally Qualified Health Ctr', 'Rural Health Clinic', 'Community Mental Health Center']
residential_terms = ['Skilled Nursing Facility', 'Nursing Facility', 'Custodial Care Facility', 'Hospice', 'Adult Living Care Facility', 'Assisted Living Facility', 'Group Home']
home_terms = ['Patient Home', 'Telehealth Provided in Pat Hm', 'Temporary Lodging', 'Place of Employment-Worksite']
emergency_terms = ['Ambulance (land)', 'Ambulance (air or water)', 'Urgent Care Facility', 'Emergency Room - Hospital']
specialty_terms = ['End-Stage Renal Disease Facil', 'Inpatient Psychiatric Facility', 'Psych Facility Partial Hosp', 'Comprehensive Inpt Rehab Fac', 'Comprehensive Outpt Rehab Fac']

for code, label in stdplac_map.items():
    if label in hospital_terms:
        stdplac_group_map[code] = "Hospital"
    elif label in clinic_terms:
        stdplac_group_map[code] = "Clinic"
    elif label in residential_terms:
        stdplac_group_map[code] = "Residential Facility"
    elif label in home_terms:
        stdplac_group_map[code] = "Home Setting"
    elif label in emergency_terms:
        stdplac_group_map[code] = "Emergency"
    elif label in specialty_terms:
        stdplac_group_map[code] = "Specialty Care"
    elif "Pharmacy" in label:
        stdplac_group_map[code] = "Pharmacy"
    elif "Mental Health" in label or "Subst Abuse" in label:
        stdplac_group_map[code] = "Behavioral Health"
    else:
        stdplac_group_map[code] = "Other/Unknown"

# Save to JSON
output_path = "stdplac_group_map.json"
with open(output_path, "w") as f:
    json.dump(stdplac_group_map, f, indent=4)


